# 09 — Guardrails & hooks

Keyword block, PII redact, pipeline hooks, text transforms, sentence chunker.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises guardrail construction, PipelineHooks, and PipelineHookExecutor.


### `guardrail()` factory


In [ ]:
from getpatter import guardrail, Guardrail
with _setup.cell('guardrail_decorator', tier=1, env=env) as ok:
    if ok:
        @guardrail
        def no_competitor_mention(text: str) -> str | None:
            """Block mentions of competitor names."""
            competitors = ['rival-co', 'othercorp']
            for c in competitors:
                if c.lower() in text.lower():
                    return f'I cannot discuss {c}.'
            return None  # allow

        result_allowed = no_competitor_mention.handler('Hello, how can I help?')
        result_blocked = no_competitor_mention.handler('Have you tried rival-co?')
        print(f'Allowed text → handler returns: {result_allowed!r}')
        print(f'Blocked text → handler returns: {result_blocked!r}')
        assert result_allowed is None
        assert result_blocked is not None
        assert isinstance(no_competitor_mention, Guardrail)


### Agent with guardrails


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, guardrail
with _setup.cell('guardrail_in_agent', tier=1, env=env) as ok:
    if ok:
        @guardrail
        def no_pricing_talk(text: str) -> str | None:
            """Redirect pricing questions to sales."""
            if 'price' in text.lower() or 'cost' in text.lower():
                return 'For pricing, please contact our sales team.'
            return None

        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000', auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        agent = p.agent(
            system_prompt='You are a support agent.',
            engine=OpenAIRealtime(api_key='sk-test'),
            guardrails=[no_pricing_talk],
        )
        print(f'Agent guardrails: {[g.name for g in agent.guardrails]}')
        assert len(agent.guardrails) == 1
        assert agent.guardrails[0].name == 'no_pricing_talk'


### PipelineHooks


In [ ]:
from getpatter import PipelineHooks, PipelineHookExecutor
with _setup.cell('pipeline_hooks', tier=1, env=env) as ok:
    if ok:
        events: list[str] = []

        async def on_transcript(text: str, is_final: bool) -> None:
            events.append(f'transcript:{text[:20]}:final={is_final}')

        async def on_llm_response(text: str) -> str:
            events.append(f'llm:{text[:20]}')
            return text  # pass-through

        hooks = PipelineHooks(
            on_transcript=on_transcript,
            on_llm_response=on_llm_response,
        )
        print(f'PipelineHooks constructed: on_transcript={hooks.on_transcript is not None}')
        print(f'                           on_llm_response={hooks.on_llm_response is not None}')
        executor = PipelineHookExecutor(hooks)
        print(f'PipelineHookExecutor: {type(executor).__name__}')
        assert hooks.on_transcript is on_transcript


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
